In [ ]:
#mount drive dan membaca data
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
path = '/content/drive/MyDrive/FOLDER/DATASET.xlsx'
df = pd.read_excel(path)
df

In [ ]:
#Import library
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import (mean_squared_error, mean_absolute_percentage_error, r2_score)

In [ ]:
#Korelasi channel dg target
korelasi = df.corr()['Y'].sort_values(ascending=False).drop('Y')
print (korelasi)

In [ ]:
# Seleksi berdasarkan nilai absolut korelasi
korelasi = df.corr()['Y'].sort_values(ascending=False).drop('Y')
corr_sorted = korelasi.reindex(
korelasi.abs().sort_values(ascending=False).index)

print("Ranking Var X:")
print(corr_sorted.abs())

In [ ]:
# Define x dan y
X = df.drop(columns=['Y'])
y = df['Y']
display(df)

# Reproducible
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

#k fold
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42)

# Save root
root_save = '/content/drive/MyDrive/Folder/Hasil Predict'
os.makedirs(root_save, exist_ok=True)

In [ ]:
# LOOP KFOLD dan
hasil = []
for n_x in range(1, 10):
    top_x = corr_sorted.index[:n_x].tolist()
    print(f"{n_x} Var X")
    print(f"FITUR : {', '.join(top_var x)}")

    #simpan data
    hasil_fold = []
    fold_train = []
    fold_test = []
    actual_train = []
    pred_train = []
    actual_test = []
    pred_test = []

    #Loop k fold
    for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

        X_train = X.iloc[train_idx][top_x]
        X_test = X.iloc[test_idx][top_x]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Fit model
        model = LinearRegression()
        model.fit(X_train, y_train)

        # Prediksi y train & test
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        # Metrik Train
        train_mse = mean_squared_error(y_train, y_pred_train)
        train_rmse = np.sqrt(train_mse)
        train_mape = mean_absolute_percentage_error(y_train, y_pred_train)*100
        train_r2 = r2_score(y_train, y_pred_train)

        # Metrik Test
        test_mse = mean_squared_error(y_test, y_pred_test)
        test_rmse = np.sqrt(test_mse)
        test_mape = mean_absolute_percentage_error(y_test, y_pred_test)*100
        test_r2 = r2_score(y_test, y_pred_test)

        # Persamaan Tiap fold
        persamaan_fold = f"y = {model.intercept_:.6f}"
        for nama, coef in zip(top_channels, model.coef_):
            if coef >= 0:
               persamaan_fold += f" + ({coef:.6f})*{nama}"
            else:
               persamaan_fold += f" - ({abs(coef):.6f})*{nama}"

        print(f"\n===== {n_channel} CHANNEL | FOLD {fold} =====")
        print("\n--- TRAIN ---")
        print(f"R2 score    : {train_r2score:.4f}")
        print(f"RMSE        : {train_rmse:.4f}")
        print(f"MSE         : {train_mse:.4f}")
        print(f"MAPE        : {train_mape:.4f}")
        print(f"Adj_R2      : {train_adj_r2:.4f}")

        print("\n--- TEST ---")
        print(f"R2 score    : {test_r2score:.4f}")
        print(f"RMSE        : {test_rmse:.4f}")
        print(f"MSE         : {test_mse:.4f}")
        print(f"MAPE        : {test_mape:.4f}")
        print(f"Adj_R2      : {test_adj_r2:.4f}")
        print("\nPersamaan:")
        print(persamaan_fold)

    #GRAFIK ACTUAL VS PREDIKSI
    # TRAIN + TEST
    plt.figure(figsize=(7,7))
    plt.scatter(
        actual_train,
        pred_train,
        color='blue',
        alpha=0.8,
        label='Train',
        edgecolors='k')
    plt.scatter(
        actual_test,
        pred_test,
        color='orange',
        alpha=0.8,
        label='Test',
        edgecolors='k')
    min_val = min(
              min(actual_train), min(actual_test),
              min(pred_train), min(pred_test))
    max_val = max(
              max(actual_train), max(actual_test),
              max(pred_train), max(pred_test))
    plt.plot(
        [min_val, max_val],
        [min_val, max_val],
        'r--',
        linewidth=2,
        label='Perfect Prediction')

    plt.xlim(min_val-0.5, max_val+0.5)
    plt.ylim(min_val-0.5, max_val+0.5)
    plt.xlabel('Aktual Y')
    plt.ylabel('Prediksi Y')
    plt.title(f'Aktual vs Prediksi {n_x}')
    plt.legend()
    plt.grid(True)
    plt.savefig(
    f"{root_save}/{n_channel}_channel/actual_vs_predicted_train_test.png",
    dpi=300,
    bbox_inches='tight')
    plt.close()

    #Hasil fold
    hasil_fold_df = pd.DataFrame(hasil_fold)
    hasil_fold_df.to_excel(
    f"{root_save}/{n_channel}_channel/hasil_fold.xlsx",
    index=False)
    hasil.extend(hasil_fold)

# HASIL AKHIR
hasil_df = pd.DataFrame(hasil)
hasil_df.to_excel(
    f"{root_save}/hasil_mlr_n_x.xlsx",
    index=False)
hasil_df